In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import numpy as np

In [ ]:
def plot_interactive_reconstructions(ds1, ds2, ds1_name="Dataset 1", ds2_name="Dataset 2",
                                     epoches=None, variables=None, samples=None):
    # Resolve selections from ds1 (assumed to share coords with ds2)
    variables = ds1.variable.values if variables is None else ds1.variable.values[variables]
    epochs    = ds1.epoch.values    if epoches  is None else ds1.epoch.values[epoches]
    samples   = ds1.sample.values   if samples  is None else np.asarray(samples)

    var_list    = variables.tolist()
    epoch_list  = epochs.tolist()
    sample_list = samples.tolist()

    # ── 1. Timestamps ─────────────────────────────────────────────────────────
    sample_dates = {}
    ts_block = ds1.timestamp.sel(sample=sample_list).compute()
    for sample in sample_list:
        ts = ts_block.sel(sample=sample).values.squeeze().item()
        sample_dates[sample] = pd.Timestamp(ts, unit='s').strftime('%Y-%m-%d %H:%M')

    # ── 2. Load both datasets ─────────────────────────────────────────────────
    print("Loading data subsets…")
    sub1 = ds1.sel(epoch=epoch_list, sample=sample_list, variable=var_list)
    sub2 = ds2.sel(epoch=epoch_list, sample=sample_list, variable=var_list)

    orig_all   = sub1.original.squeeze(["time", "ensemble"]).compute()
    recon1_all = sub1.reconstruction.squeeze(["time", "ensemble"]).compute()
    recon2_all = sub2.reconstruction.squeeze(["time", "ensemble"]).compute()

    orig_np   = orig_all.values.astype(np.float32)
    recon1_np = recon1_all.values.astype(np.float32)
    recon2_np = recon2_all.values.astype(np.float32)

    var_idx   = {v: i for i, v in enumerate(var_list)}
    samp_idx  = {s: i for i, s in enumerate(sample_list)}
    epoch_idx = {e: i for i, e in enumerate(epoch_list)}

    # ── 3. Colour scales ──────────────────────────────────────────────────────
    print("Computing colour scales…")
    var_scales  = {}
    diff1_scales = {}
    diff2_scales = {}
    for var in var_list:
        vi = var_idx[var]
        o  = orig_np [:, :, vi]
        r1 = recon1_np[:, :, vi]
        r2 = recon2_np[:, :, vi]
        var_scales[var]   = (float(min(o.min(), r1.min(), r2.min())),
                             float(max(o.max(), r1.max(), r2.max())))
        diff1_scales[var] = (-float(np.abs(o - r1).max()), float(np.abs(o - r1).max()))
        diff2_scales[var] = (-float(np.abs(o - r2).max()), float(np.abs(o - r2).max()))

    # ── 4. Plot loop ──────────────────────────────────────────────────────────
    print("Plotting…")
    for epoch in epoch_list:
        ei = epoch_idx[epoch]

        for var in var_list:
            vi         = var_idx[var]
            vmin, vmax = var_scales[var]
            cmap_main  = 'Blues' if var == 'tp' else 'magma'
            n          = len(sample_list)

            # 4 columns: Original | ds1_name | ds2_name | Diff ds1 | Diff ds2
            fig, axes = plt.subplots(n, 5, figsize=(20, 4 * n), constrained_layout=True)
            if n == 1:
                axes = np.expand_dims(axes, axis=0)

            im_main = im_diff1 = im_diff2 = None

            for s_idx, sample in enumerate(sample_list):
                si     = samp_idx[sample]
                orig   = np.flipud(orig_np  [ei, si, vi].astype(np.float64))
                recon1 = np.flipud(recon1_np[ei, si, vi].astype(np.float64))
                recon2 = np.flipud(recon2_np[ei, si, vi].astype(np.float64))
                diff1  = orig - recon1
                diff2  = orig - recon2
                date_str = sample_dates[sample]

                rmse1 = float(np.sqrt(np.mean(diff1 ** 2)))
                rmse2 = float(np.sqrt(np.mean(diff2 ** 2)))

                # Shared diff scale across both datasets for fair comparison
                dabs  = max(np.abs(diff1).max(), np.abs(diff2).max())
                dmin, dmax = -float(dabs), float(dabs)

                # Col 0 — Original
                ax = axes[s_idx, 0]
                im_main = ax.imshow(orig, cmap=cmap_main, vmin=vmin, vmax=vmax)
                ax.set_title(f"Original\n{var} | {date_str}", fontsize=9)
                ax.axis('off')

                # Col 1 — ds1 reconstruction
                ax = axes[s_idx, 1]
                ax.imshow(recon1, cmap=cmap_main, vmin=vmin, vmax=vmax)
                ax.set_title(f"{ds1_name}\n{var}", fontsize=9)
                ax.axis('off')

                # Col 2 — ds2 reconstruction
                ax = axes[s_idx, 2]
                ax.imshow(recon2, cmap=cmap_main, vmin=vmin, vmax=vmax)
                ax.set_title(f"{ds2_name}\n{var}", fontsize=9)
                ax.axis('off')

                # Col 3 — Difference ds1
                ax = axes[s_idx, 3]
                im_diff1 = ax.imshow(diff1, cmap='RdBu_r', vmin=dmin, vmax=dmax)
                ax.set_title(f"Diff (Original − {ds1_name})\nRMSE: {rmse1:.4f}", fontsize=9)
                ax.axis('off')

                # Col 4 — Difference ds2
                ax = axes[s_idx, 4]
                im_diff2 = ax.imshow(diff2, cmap='RdBu_r', vmin=dmin, vmax=dmax)
                ax.set_title(f"Diff (Original − {ds2_name})\nRMSE: {rmse2:.4f}", fontsize=9)
                ax.axis('off')

            # Shared colourbar for the three field columns
            fig.colorbar(im_main,  ax=axes[:, :3], orientation='vertical',
                         fraction=0.02, pad=0.04).set_label(f'Value scale — {var}')
            # Shared colourbar for both difference columns
            fig.colorbar(im_diff1, ax=axes[:, 3:], orientation='vertical',
                         fraction=0.04, pad=0.04).set_label(f'Difference scale — {var}')

            plt.show()

In [ ]:
import xarray as xr


datasets = [
   {'name': 'Symetric Range Normalization', 'path':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae-answer/mse_kl_04_0_20260417_092115/samples.zarr',
   },
   {'name': 'z-Score+std', 'path':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/13-qrl-vae-winds-focal/13-qrl-vae-winds-focal-z64x16x16-20260609_150346/samples.zarr',
   },
 
 ]



In [ ]:
ds1 = xr.open_zarr(datasets[0]['path'], zarr_format=3, consolidated=True)
ds2 = xr.open_zarr(datasets[1]['path'], zarr_format=3, consolidated=True)
ds1_name = datasets[0]['name']
ds2_name = datasets[1]['name']

 
plot_interactive_reconstructions(ds1, ds2, ds1_name=ds1_name, ds2_name=ds2_name,
                                     epoches=[-1], variables=None, samples=[0])

    

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[1])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[2])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[3])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[4])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[5])
    except Exception as e:
        print(f"  ERROR: {e}")